In [ ]:
import numpy as np
from coupled_solver.utils.get_data import read_tinker

x_q, q, d, Q, alpha, mass, polar_group, thole, \
               connections_12, connections_13, \
               pointer_connections_12, pointer_connections_13, \
               p12scale, p13scale, N = read_tinker('molecules/1pgb/1pgb', float)

Reading parameters from ../amoebapro13.prm


In [19]:
rng = np.random.default_rng()

In [20]:
x = rng.random(size=(30000,3)) # 300.000 puntos aleatorios

In [21]:
ep_in = 1.
result_vec = np.zeros((30000))
result_esc = np.zeros((30000))

In [22]:
from numba import njit

@njit
def charges_fun_vec(x, x_q, q, d, Q):
    T2 = np.zeros((len(x_q),3,3))
    dist = x - x_q
    norm = np.sqrt(np.sum((dist*dist), axis = 1))
    T0 = 1/norm[:]
    T1 = np.transpose(dist.transpose()/norm**3)
    T2[:,:,:] = np.ones((len(x_q),3,3))[:]*dist.reshape((len(x_q),1,3))*np.transpose(np.ones((len(x_q),3,3))*dist.reshape((len(x_q),1,3)), (0,2,1))/norm.reshape((len(x_q),1,1))**5
    phi_c = np.sum(q[:]*T0[:]) + np.sum(T1[:]*d[:]) + 0.5*np.sum(np.sum(T2[:]*Q[:],axis=1))
    return phi_c/(4*np.pi*ep_in)

@njit
def charges_fun_esc(x, x_q, q, d, Q):
    phi_c = 0.0
    for i in range(len(x_q)):
        dx = x[0] - x_q[i, 0]
        dy = x[1] - x_q[i, 1]
        dz = x[2] - x_q[i, 2]
        norm = np.sqrt(dx*dx + dy*dy + dz*dz)
        if norm>1e-10:
            T0 = 1/norm
            phi_c += q[i] * T0

            T1_x = dx/(norm**3)
            T1_y = dy/(norm**3)
            T1_z = dz/(norm**3)
            phi_c += T1_x*d[i,0] + T1_y*d[i,1] + T1_z*d[i,2]

            T2_xx = (dx * dx)/(2*norm**5)
            T2_xy = (dx * dy)/(2*norm**5)
            T2_xz = (dx * dz)/(2*norm**5)
            T2_yx = (dy * dx)/(2*norm**5)
            T2_yy = (dy * dy)/(2*norm**5)
            T2_yz = (dy * dz)/(2*norm**5)
            T2_zx = (dz * dx)/(2*norm**5)
            T2_zy = (dz * dy)/(2*norm**5)
            T2_zz = (dz * dz)/(2*norm**5)
            phi_c += T2_xx*Q[i,0,0] + T2_xy*Q[i,0,1] + T2_xz*Q[i,0,2]
            phi_c += T2_yx*Q[i,1,0] + T2_yy*Q[i,1,1] + T2_yz*Q[i,1,2]
            phi_c += T2_zx*Q[i,2,0] + T2_zy*Q[i,2,1] + T2_zz*Q[i,2,2]
    return phi_c/(4*np.pi*ep_in)


In [23]:
# Compilación rápida

dummy = np.array([0., 0., 0.])
dummyx = np.zeros((1,3))
dummyq = np.zeros((1))
dummyQ = np.zeros((1,3,3))

charges_fun_vec(dummyx, dummyx, dummyq, dummyx, dummyQ)
charges_fun_esc(dummy, dummyx, dummyq, dummyx, dummyQ)

0.0

In [24]:
import time

time_init_vec = time.time()
for j in range(len(x)):
    result_vec[j] = charges_fun_vec(x[j], x_q, q, d, Q)
time_final_vec = time.time()
time_init_esc = time.time()
for j in range(len(x)):
    result_esc[j] = charges_fun_esc(x[j], x_q, q, d, Q)
time_final_esc = time.time()

print(f'Tiempo en función vectorizada: {time_final_vec - time_init_vec} [s]')
print(f'Tiempo en función escalar: {time_final_esc - time_init_esc} [s]')


Tiempo en función vectorizada: 9.327876806259155 [s]
Tiempo en función escalar: 1.4091072082519531 [s]
